In [ ]:
from dotenv import load_dotenv

load_dotenv()

from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

video_id = 'SfOaZIGJ_gs'

transcript = YouTubeTranscriptApi().fetch(video_id)
transcripts_text = " ".join(snippet.text for snippet in transcript)

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(transcripts_text)

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = FAISS.from_texts(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

prompt = PromptTemplate(
    template="""You are a helpful assistant. Answer from the following transcript context.
If the context is insufficient, directly say "I don't know."
Use the conversation history only to understand follow-up questions. Do not invent facts.

Conversation history:
{history}

Transcript context:
{context}

Question: {question}""",
    input_variables=['history', 'context', 'question']
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
conversation_history = []

while True:
    question = input("\nYou: ").strip()
    if question.lower() in {"exit", "quit", "q"}:
        print("Chat ended.")
        break
    if not question:
        continue

    retrieved_docs = retriever.invoke(question)
    context_text = " ".join(doc.page_content for doc in retrieved_docs)
    history_text = "\n".join(
        f"User: {previous_question}\nAssistant: {previous_answer}"
        for previous_question, previous_answer in conversation_history
    ) or "No previous conversation."

    formatted_prompt = prompt.format(
        history=history_text,
        context=context_text,
        question=question,
    )
    response = llm.invoke(formatted_prompt)
    answer = response.content
    print(f"Assistant: {answer}")
    conversation_history.append((question, answer))

c:\Users\pushp\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\pushp\AppData\Local\Temp\ipykernel_12284\3636298914.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Yes, the video discusses how to start a company, including considerations for young entrepreneurs, the importance of long-term thinking, and the need for good partnerships, especially in fields like robotics.
